# 01. Exploratory Data Analysis (EDA) — IEEE-CIS Fraud Detection

### FraudSentinel Banking Fraud Detection System

This notebook performs initial exploratory data analysis on the **IEEE-CIS Fraud Detection Dataset** to investigate:
1. **Fraud vs Non-Fraud Class Distribution** (imbalance analysis)
2. **Transaction Amount Analysis** (distribution, log-scaling, fraud rates by amount)
3. **Temporal & Time-Series Patterns** (transaction hour of day, daily cycles)
4. **Categorical Feature Analysis** (card networks, product codes, device types, email domains)
5. **Data Quality & Missingness Patterns**
6. **Key Insights for Feature Engineering & Pipeline Design**

In [ ]:
# Imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', 50)

print('Libraries imported successfully!')

## 1. Data Ingestion & Dataset Overview
We inspect `data/raw/ieee_cis/train_transaction.csv` and `data/raw/ieee_cis/train_identity.csv`. If raw files are not yet present, sample data is generated for demonstration.

In [ ]:
raw_dir = Path('../data/raw/ieee_cis')
trans_path = raw_dir / 'train_transaction.csv'
id_path = raw_dir / 'train_identity.csv'

if trans_path.exists():
    print(f'Loading transaction dataset from {trans_path}...')
    df_trans = pd.read_csv(trans_path)
    if id_path.exists():
        df_id = pd.read_csv(id_path)
        df = pd.merge(df_trans, df_id, on='TransactionID', how='left')
    else:
        df = df_trans
else:
    print('Raw dataset not found at target directory. Generating synthetic demonstration dataset matching IEEE-CIS schema...')
    np.random.seed(42)
    n = 10000
    df = pd.DataFrame({
        'TransactionID': np.arange(2987000, 2987000 + n),
        'isFraud': np.random.choice([0, 1], size=n, p=[0.965, 0.035]),
        'TransactionDT': np.random.randint(86400, 86400 * 30, size=n),
        'TransactionAmt': np.random.exponential(scale=135, size=n).round(2),
        'ProductCD': np.random.choice(['W', 'H', 'C', 'S', 'R'], size=n, p=[0.75, 0.10, 0.08, 0.04, 0.03]),
        'card1': np.random.randint(1000, 18000, size=n),
        'card4': np.random.choice(['visa', 'mastercard', 'american express', 'discover'], size=n, p=[0.65, 0.28, 0.05, 0.02]),
        'card6': np.random.choice(['debit', 'credit'], size=n, p=[0.74, 0.26]),
        'P_emaildomain': np.random.choice(['gmail.com', 'yahoo.com', 'hotmail.com', 'anonymous.com', 'aol.com', None], size=n, p=[0.45, 0.25, 0.15, 0.05, 0.05, 0.05]),
        'DeviceType': np.random.choice(['desktop', 'mobile', None], size=n, p=[0.35, 0.25, 0.40]),
    })

print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

## 2. Target Variable Analysis (Fraud vs Non-Fraud)
Fraud detection datasets suffer from severe class imbalance. We analyze the exact distribution.

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_pcts = df['isFraud'].value_counts(normalize=True) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
sns.barplot(x=['Legitimate (0)', 'Fraudulent (1)'], y=fraud_counts.values, palette=['#2b5c8f', '#d9534f'], ax=ax1)
ax1.set_title('Transaction Count by Class', fontsize=13, fontweight='bold')
ax1.set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    ax1.text(i, v + (v * 0.01), f'{v:,}', ha='center', fontweight='bold')

# Pie chart
ax2.pie(fraud_counts.values, labels=['Legitimate', 'Fraudulent'], autopct='%1.2f%%', colors=['#2b5c8f', '#d9534f'], explode=(0, 0.1), startangle=140)
ax2.set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Legitimate Transactions: {fraud_counts.get(0, 0):,} ({fraud_pcts.get(0, 0):.2f}%)')
print(f'Fraudulent Transactions: {fraud_counts.get(1, 0):,} ({fraud_pcts.get(1, 0):.2f}%)')

## 3. Transaction Amount Analysis
Examining transaction amounts for legitimate vs fraudulent transactions.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Raw Amount Boxplot (log scale for visibility)
sns.boxplot(data=df, x='isFraud', y='TransactionAmt', palette=['#2b5c8f', '#d9534f'], ax=ax1)
ax1.set_yscale('log')
ax1.set_title('Transaction Amount (Log Scale) by Class', fontsize=12, fontweight='bold')
ax1.set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'])

# Log-transformed Amount Distribution
df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
sns.kdeplot(data=df[df['isFraud'] == 0], x='TransactionAmt_log', label='Legitimate', color='#2b5c8f', shade=True, ax=ax2)
sns.kdeplot(data=df[df['isFraud'] == 1], x='TransactionAmt_log', label='Fraudulent', color='#d9534f', shade=True, ax=ax2)
ax2.set_title('Density of Log(Transaction Amount)', fontsize=12, fontweight='bold')
ax2.set_xlabel('log(1 + TransactionAmt)')
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Temporal Patterns (Transaction Hour of Day)
Analyzing transaction volume and fraud rates across 24 hours of the day.

In [ ]:
df['Transaction_hour'] = (df['TransactionDT'] // 3600) % 24

hourly_summary = df.groupby('Transaction_hour')['isFraud'].agg(['count', 'sum', 'mean']).reset_index()
hourly_summary['fraud_pct'] = hourly_summary['mean'] * 100

fig, ax1 = plt.subplots(figsize=(12, 5))

color1 = '#2b5c8f'
color2 = '#d9534f'

ax1.set_xlabel('Hour of Day (0-23)', fontweight='bold')
ax1.set_ylabel('Total Transaction Volume', color=color1, fontweight='bold')
ax1.bar(hourly_summary['Transaction_hour'], hourly_summary['count'], color=color1, alpha=0.6)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
ax2.set_ylabel('Fraud Rate (%)', color=color2, fontweight='bold')
ax2.plot(hourly_summary['Transaction_hour'], hourly_summary['fraud_pct'], color=color2, linewidth=2.5, marker='o')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Transaction Volume & Fraud Rate by Hour of Day', fontsize=13, fontweight='bold')
plt.grid(False)
plt.show()

## 5. Categorical Risk Drivers (Card Network & Device Type)

In [ ]:
if 'card4' in df.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    card_fraud = df.groupby('card4')['isFraud'].agg(['count', 'mean']).reset_index()
    card_fraud['fraud_pct'] = card_fraud['mean'] * 100
    
    sns.barplot(data=card_fraud, x='card4', y='count', palette='Blues_d', ax=ax1)
    ax1.set_title('Transactions by Card Network', fontweight='bold')
    ax1.set_ylabel('Count')
    
    sns.barplot(data=card_fraud, x='card4', y='fraud_pct', palette='Reds_d', ax=ax2)
    ax2.set_title('Fraud Rate (%) by Card Network', fontweight='bold')
    ax2.set_ylabel('Fraud Percentage')
    
    plt.tight_layout()
    plt.show()

## 6. Missingness Analysis
Evaluating missing value proportions per feature column.

In [ ]:
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    plt.figure(figsize=(10, 4))
    sns.barplot(x=missing.index[:15], y=missing.values[:15], palette='Oranges_d')
    plt.title('Top Columns by Missing Value Percentage', fontweight='bold')
    plt.ylabel('Missing Percentage (%)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values found in sample dataset!')

## 7. Summary & Key Findings for Pipeline Design
- **Severe Class Imbalance**: Fraud rate is low (~3.5%), requiring evaluation metrics like PR-AUC and ROC-AUC rather than raw accuracy.
- **Amount Scaling**: Transaction amount is highly right-skewed; `log1p` transformation and decimal extraction provide strong features.
- **Time Cyclicality**: Fraud occurrence varies significantly by transaction hour.
- **Missing Value Indicators**: Missingness itself is informative in financial data; row-level missingness counts (`null_count`) will be incorporated.